# Import

In [ ]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)

# Download data

In [ ]:
noncurated_path = "../non_curated/h5ad/norman_2019_raw.h5ad"
download_file(
    url="https://exampledata.scverse.org/pertpy/norman_2019_raw.h5ad",
    dest_path=noncurated_path
)

# Initialise the dataset object

In [ ]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

In [ ]:
cur_data.adata.obs

# OBS slot curation

### Drop all columns starting with `guide_`

In [ ]:
cur_data.adata.obs = cur_data.adata.obs[['guide_identity', 'guide_ids', 'gemgroup']]

### Add cell barcodes to the obs slot

In [ ]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.astype(str)

print(cur_data.adata.obs[['cell_barcode']].head())

### Show unique perturbations

In [ ]:
cur_data.show_unique(slot = 'obs', column = 'guide_identity')

### Rename `guide_identity` to `perturbation_name`

In [ ]:
cur_data.rename_columns(slot = 'obs', name_dict = {'guide_identity': 'perturbation_name'})

In [ ]:
# split perturbation_name to keep the first part only
cur_data.adata.obs['perturbation_name'] = cur_data.adata.obs['perturbation_name'].str.split('__').str[0]
cur_data.adata.obs[['perturbation_name']]

In [ ]:
cur_data.adata.obs

### Add guide RNA information

In [ ]:
# download the guide RNA spreadsheet
download_file(
    url="http://www.science.org/doi/suppl/10.1126/science.aax4438/suppl_file/aax4438_tables2.xlsx",
    dest_path="../supplementary/norman_2019_guide_info.xlsx"
)

# read in the guide RNA spreadsheet
# guides for the K562 essential day 6 library are in "TabB_K562_day6_library"
guide_info_df = pd.read_excel("../supplementary/norman_2019_guide_info.xlsx", sheet_name="Perturbseq_sgRNA_info")

# create perturbation_name column in guide_info_df
guide_info_df['perturbation_name'] = guide_info_df['gene_A'] + '_' + guide_info_df['gene_B']
# check that all perturbation names in cur_data are in guide_info_df
print(f"All perturbation names in cur_data are in guide_info_df: {cur_data.adata.obs['perturbation_name'].isin(guide_info_df['perturbation_name']).all()}")
# create guide_sequence column in guide_info_df
guide_info_df['guide_sequence'] = guide_info_df['protospacer_sequence_A'] + '|' + guide_info_df['protospacer_sequence_B']
# subset for necessary columns
guide_info_df = guide_info_df[['perturbation_name', 'guide_sequence']]
# merge cur_data.adata.obs with guide_info_df on perturbation_name
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='perturbation_name', how='left')
# # check that there are no missing guide sequences
print(f"Number of missing guide sequences: {cur_data.adata.obs['guide_sequence'].isna().sum()}")

### Standardise perturbation targets

In [ ]:
cur_data.adata.obs['target'] = cur_data.adata.obs['perturbation_name'].str.replace('_', '|').str.replace(r'NegCtrl\d+', 'control_nontargeting', regex=True)
cur_data.adata.obs[['target']]

In [ ]:
cur_data.standardize_genes(
    slot='obs',
    input_column='target',
    input_column_type='gene_symbol',
    multiple_entries=True,
    multiple_entries_sep='|'
)

### Add `perturbed_target_number` column

In [ ]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

### Encode chromosomes as integers

In [ ]:
cur_data.chromosome_encoding()

### Assign replicates

In [ ]:
cur_data.adata.obs = cur_data.adata.obs.rename(columns={'gemgroup': 'technical_replicate'})

In [ ]:
cur_data.adata.obs

### Add metadata

In [ ]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        #----- dataset -----#
        "dataset_id": cur_data.dataset_id,
        #----- sample -----#
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        #----- perturbation type -----#
        "perturbation_type_label": "CRISPRa",
        "perturbation_type_id": None,
        #----- data modality -----#
        "data_modality": "Perturb-seq", # different from "method_name_label"; more general term - choice of CRISPR, MAVE and Perturb-seq
        #----- significance -----#
        "significant": None,
        "significance_criteria": None,
        #----- score interpretation -----#
        "score_interpretation": None,
        #----- treatment -----#
        "treatment_label": None,
        "treatment_id": None,
        #----- replicate -----#
        # "technical_replicate": None,
        "biological_replicate": None,
        #----- model system -----#
        "model_system_label": "cell_line",
        "model_system_id": None,
        #----- tissue -----#
        "tissue": "blood",
        #----- cell line -----#
        "cell_line_label": "K 562 cell",
        "cell_line_id": None,
        #----- cell type -----#
        "cell_type_label": "lymphoblast",
        "cell_type_id": None,
        #----- disease -----#
        "disease_label": "chronic myelogenous leukemia, BCR-ABL1 positive",
        "disease_id": None,
        #----- timepoint -----#
        "timepoint": "P7DT0H0M0S",
        #----- species -----#
        "species": "Homo sapiens",
        #----- sex -----#
        "sex_label": "female",
        "sex_id": None,
        #----- developmental stage -----#
        "developmental_stage_label": "adult",
        "developmental_stage_id": None,
        #----- study metadata -----#
        "study_title": "Exploring genetic interaction manifolds constructed from rich single-cell phenotypes",
        "study_uri": "https://doi.org/10.1126/science.aax4438",
        "study_year": 2019,
        #----- authors -----#
        "first_author": "Thomas M. Norman",
        "last_author": "Jonathan S. Weissman",
        #----- experiment metadata -----#
        "experiment_title": "Perturb-seq CRISPRa of K562 cells to explore genetic interaction manifolds",
        "experiment_summary": """
            K562 cells were engineered by lentiviral transduction of dox-inducible CRISPRa SunTag system. The sgRNA library consisting of 295 dual sgRNA vectors was packaged into lentiviral particles, spinfected into K562 cells. Cells were grown until day 7, at which point they were harvested, processed using Chromium Single Cell 3-prime Gel Beads v2 kit and sequenced on an Illumina NovaSeq 6000.
        """,
        #----- number of perturbed targets/samples -----#
        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],
        #----- library generation type -----#
        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",
        #----- library generation method -----#
        "library_generation_method_id": "EFO:0022898",
        "library_generation_method_label": "dCas9-Suntag",
        #----- enzyme and library delivery method -----#
        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",
        #----- enzyme and library integration state -----#
        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",
        #----- enzyme and library expression control -----#
        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "inducible transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",
        #----- library name and URI and manufacturer -----#
        "library_name": "custom",
        "library_uri": None,
        "library_manufacturer": "Weissman lab",
        #----- library format -----#
        "library_format_id": None,
        "library_format_label": "pooled",
        #----- library scope -----#
        "library_scope_id": None,
        "library_scope_label": "focused",
        #----- library perturbation type -----#
        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "activation",
        #----- library additional metadata -----#
        "library_lentiviral_generation": "3",
        "library_grnas_per_target": "1",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()), # for CRISPR/Perturb-seq
        "library_total_variants": None, # for MAVE
        #----- readout dimensionality -----#
        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",
        #---- readout type -----#
        "readout_type_id": None,
        "readout_type_label": "transcriptomic",
        #----- readout technology -----#
        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",
        #----- method -----#
        "method_name_id": None,
        "method_name_label": "Perturb-seq", # different from "data_modality"; more specific term - specific name of the technique
        "method_uri": None,
        #----- sequencing library kit -----#
        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "10x Genomics Single Cell 3-prime v2",
        #----- sequencing platform -----#
        "sequencing_platform_id": None,
        "sequencing_platform_label": "Illumina NovaSeq 6000",
        #----- sequencing strategy -----#
        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",
        #----- software used for counts-----#
        "software_counts_id": None,
        "software_counts_label": "CellRanger",
        #----- software used for analysis -----#
        "software_analysis_id": None,
        "software_analysis_label": "custom",
        #----- reference genome -----#
        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",
        #----- license -----#
        "license_label": "free to use license",
        "license_id": "SWO:1000061",
        #----- external datasets -----#
        "associated_datasets": json.dumps([
            {
                "dataset_accession": "GSE133344",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE133344",
                "dataset_description": "Raw counts; matrix.mtx, features.tsv, barcodes.tsv",
                "dataset_file_name": "GSE278572_*.*",
            },
            {
                "dataset_accession": "norman_2019_raw.h5ad",
                "dataset_uri": "https://exampledata.scverse.org/pertpy/norman_2019_raw.h5ad",
                "dataset_description": "Raw counts - .h5ad file from pertpy",
                "dataset_file_name": "norman_2019_raw.h5ad",
            }
        ])
    }
)

In [ ]:
cur_data.adata.obs

### Curate tissue information


In [ ]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

### Curate cell type information

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

### Curate cell line information

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_line_label',
    column_type='term_name',
    ontology_type='cell_line',
    overwrite=True
)

### Curate disease information

In [ ]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs', verbose=True)

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.adata.var

In [ ]:
cur_data.create_columns(
    slot = 'var',
    col_dict={'gene_ensembl_id': cur_data.adata.var.index},
    overwrite=True
)

In [ ]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ensembl_id",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

### Replace unmapped gene symbols with original gene symbols

In [ ]:
cur_data.adata.var['gene_symbol'] = cur_data.adata.var['gene_symbol'].fillna(
    cur_data.adata.var['gene_symbols']
)

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

# Save the dataset

In [ ]:
cur_data.save_curated_data_h5ad()

In [ ]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

# Upload to BigQuery

In [ ]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/norman_2019_raw_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

# Upload to GC Storage

In [ ]:
!gcloud storage cp ../curated/h5ad/norman_2019_raw_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/